# Japanese Real Estate Transaction Data — Cleaning Pipeline
## Preparing MLIT Transaction Data for Real Estate Analysis

**Data source:** Japanese Ministry of Land, Infrastructure, Transport and Tourism (MLIT)  
**Dataset:** Real Estate Transaction Price Information, 1945–2020  
**Geography:** Chiba Prefecture, Japan  
**Source file:** [Japan Real Estate Transaction Prices — Kaggle](https://www.kaggle.com/datasets/nishiodens/japan-real-estate-transaction-prices?select=prefecture_code.csv)

---

### Purpose

This notebook prepares MLIT real estate transaction data for downstream analysis. It provides a reusable data-cleaning workflow that can support different analytical projects without tying preprocessing decisions to a particular prediction target or machine-learning model.

### Approach

The workflow examines the dataset's structure, data types, distributions, and missing values; investigates data-quality issues; standardizes fields where appropriate; and creates generally useful features where their definitions are supported by the available data.

Missing values are assessed in context rather than automatically replaced. Cleaning decisions aim to preserve the meaning of the original records and distinguish genuine values from unavailable or inapplicable information.

### Output

A cleaned dataset exported to CSV for reuse in subsequent notebooks and analytical workflows.

---

*The source data was downloaded from Kaggle and originates from MLIT's Real Estate Transaction Price Information. This dataset covers transactions from 1945 through 2020. Analyses requiring more recent market conditions will need additional data beyond 2020.*

### Original Feature Dictionary

| Feature | Description | Data Type |
|---|---|---|
| `No` | Record identifier in the dataset | Integer |
| `Type` | Real estate transaction/property type | Categorical |
| `Region` | Characteristics of the surrounding area | Categorical |
| `MunicipalityCode` | Municipality code | Integer |
| `Prefecture` | Prefecture name | Categorical |
| `Municipality` | Municipality name | Categorical |
| `DistrictName` | District name | Categorical |
| `NearestStation` | Name of the nearest station | Categorical |
| `TimeToNearestStation` | Original time-to-station field, which may contain ranges or non-numeric text | Object |
| `MinTimeToNearestStation` | Minimum estimated time to the nearest station, in minutes | Float |
| `MaxTimeToNearestStation` | Maximum estimated time to the nearest station, in minutes | Float |
| `TradePrice` | Transaction price, in yen | Integer |
| `FloorPlan` | Property floor-plan description | Categorical |
| `Area` | Surveyed land area, in m² | Integer |
| `AreaIsGreaterFlag` | Indicates land area recorded as 2,000 m² or greater | Integer |
| `UnitPrice` | Unit land price, in yen per m² | Float |
| `PricePerTsubo` | Unit land price, in yen per tsubo | Float |
| `LandShape` | Shape of the land | Categorical |
| `Frontage` | Road frontage length, in meters | Float |
| `FrontageIsGreaterFlag` | Indicates frontage recorded as 50 m or longer | Boolean |
| `TotalFloorArea` | Total building floor area, in m² | Float |
| `TotalFloorAreaIsGreaterFlag` | Indicates floor area recorded as 2,000 m² or greater | Integer |
| `BuildingYear` | Year of building construction | Float |
| `PrewarBuilding` | Flag for buildings recorded as constructed before 1945 | Integer |
| `Structure` | Building structure, potentially containing multiple structure categories | Categorical |
| `Use` | Recorded current use of the property | Categorical |
| `Purpose` | Recorded intended future use of the property | Categorical |
| `Direction` | Direction of the frontage road | Categorical |
| `Classification` | Classification of the frontage road | Categorical |
| `Breadth` | Frontage road width, in meters | Float |
| `CityPlanning` | Land-use zoning or planning designation | Categorical |
| `CoverageRatio` | Maximum permitted building coverage ratio, in percent | Float |
| `FloorAreaRatio` | Maximum permitted floor-area ratio, in percent | Float |
| `Period` | Transaction period | Categorical |
| `Year` | Transaction year | Integer |
| `Quarter` | Transaction quarter | Integer |
| `Renovation` | Recorded renovation status | Categorical |
| `Remarks` | Additional transaction notes | Categorical/text |


In [ ]:
# =============================
# Libraries
# =============================

# Data manipulation
import pandas as pd
import numpy as np



## 1. Data Loading

MLIT publishes real estate transaction data in separate CSV files for each prefecture (`01.csv` through `47.csv`). Chiba Prefecture corresponds to `12.csv`. Loading only this file limits processing to the transactions relevant to this analysis, rather than loading the approximately 3.9 million records in the nationwide dataset. Users of this notebook may load data for other prefectures by putting that prefecture's csv file in the same folder as the notebook and changing the file name loaded in the cell below.

In [ ]:
### Load Chiba Prefecture transaction data only (Prefecture code 12)
# Full dataset covers 47 prefectures — Chiba is file 12.csv
# Source: MLIT Land General Information System (土地総合情報システム)


real_estate_df = pd.read_csv("12.csv")

print(f"Rows loaded: {len(real_estate_df):,}")
print(f"Prefecture: {real_estate_df['Prefecture'].unique()}")



Rows loaded: 184,548
Prefecture: ['Chiba Prefecture']


## 2. Data Inspection

In [ ]:
# Preview data
print(real_estate_df.shape)
pd.set_option('display.max_columns', None)
real_estate_df.head(20)

(184548, 38)


,No,Type,Region,MunicipalityCode,Prefecture,Municipality,DistrictName,NearestStation,TimeToNearestStation,MinTimeToNearestStation,MaxTimeToNearestStation,TradePrice,FloorPlan,Area,AreaIsGreaterFlag,UnitPrice,PricePerTsubo,LandShape,Frontage,FrontageIsGreaterFlag,TotalFloorArea,TotalFloorAreaIsGreaterFlag,BuildingYear,PrewarBuilding,Structure,Use,Purpose,Direction,Classification,Breadth,CityPlanning,CoverageRatio,FloorAreaRatio,Period,Year,Quarter,Renovation,Remarks
0,1,Residential Land(Land Only),Residential Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Soga,28,28.0,28.0,25000000,NaN,130,0,190000.0,640000.0,Rectangular Shaped,9.0,False,NaN,0,NaN,0,NaN,NaN,NaN,West,City Road,6.0,Category I Exclusively Low-story Residential Zone,60.0,150.0,1st quarter 2018,2018,1,NaN,NaN
1,2,Residential Land(Land Only),Residential Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Soga,28,28.0,28.0,25000000,NaN,130,0,190000.0,640000.0,Rectangular Shaped,9.0,False,NaN,0,NaN,0,NaN,NaN,NaN,West,City Road,6.0,Category I Exclusively Low-story Residential Zone,60.0,150.0,1st quarter 2018,2018,1,NaN,NaN
2,3,Residential Land(Land Only),Residential Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Soga,28,28.0,28.0,28000000,NaN,210,0,130000.0,450000.0,Irregular Shaped,13.5,False,NaN,0,NaN,0,NaN,NaN,NaN,North,City Road,6.0,Category I Exclusively Low-story Residential Zone,60.0,150.0,4th quarter 2017,2017,4,NaN,NaN
3,4,"Pre-owned Condominiums, etc.",NaN,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Chiba,30-60minutes,30.0,60.0,18000000,4LDK,100,0,NaN,NaN,NaN,NaN,False,NaN,0,1998.0,0,RC,NaN,House,NaN,NaN,NaN,Category II Residential Zone,60.0,200.0,1st quarter 2019,2019,1,Done,NaN
4,5,Residential Land(Land and Building),Residential Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Chiba,30-60minutes,30.0,60.0,40000000,NaN,170,0,NaN,NaN,Irregular Shaped,15.8,False,135.0,0,2008.0,0,W,House,House,Southwest,City Road,5.0,Category II Exclusively Medium-high Residentia...,60.0,200.0,3rd quarter 2018,2018,3,NaN,NaN
5,6,Residential Land(Land and Building),Residential Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Chiba,30-60minutes,30.0,60.0,38000000,NaN,180,0,NaN,NaN,Rectangular Shaped,12.0,False,155.0,0,1998.0,0,W,House,House,West,City Road,6.0,Category I Exclusively Low-story Residential Zone,60.0,150.0,1st quarter 2017,2017,1,NaN,NaN
6,7,Residential Land(Land Only),Commercial Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Chiba,30-60minutes,30.0,60.0,37000000,NaN,280,0,130000.0,440000.0,Irregular Shaped,17.3,False,NaN,0,NaN,0,NaN,NaN,House,Southwest,City Road,15.0,Category II Exclusively Medium-high Residentia...,60.0,200.0,3rd quarter 2016,2016,3,NaN,NaN
7,8,Residential Land(Land and Building),Commercial Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Chiba,30-60minutes,30.0,60.0,120000000,NaN,280,0,NaN,NaN,Irregular Shaped,17.3,False,360.0,0,NaN,0,NaN,NaN,House,Southwest,City Road,15.0,Category II Exclusively Medium-high Residentia...,60.0,200.0,3rd quarter 2016,2016,3,NaN,NaN
8,9,Residential Land(Land Only),Residential Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Chiba,30-60minutes,30.0,60.0,31000000,NaN,220,0,140000.0,460000.0,Rectangular Shaped,13.0,False,NaN,0,NaN,0,NaN,NaN,House,Southwest,City Road,6.0,Category I Exclusively Low-story Residential Zone,60.0,150.0,2nd quarter 2015,2015,2,NaN,NaN
9,10,"Pre-owned Condominiums, etc.",NaN,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Aobacho,Chiba,30-60minutes,30.0,60.0,12000000,3LDK,95,0,NaN,NaN,NaN,NaN,False,NaN,0,1998.0,0,RC,House,House,NaN,NaN,NaN,Category II Residential Zone,60.0,200.0,2nd quarter 2015,2015,2,NaN,Dealings in which auction or arbiter participates


In [ ]:
real_estate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184548 entries, 0 to 184547
Data columns (total 38 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   No                           184548 non-null  int64  
 1   Type                         184548 non-null  object 
 2   Region                       137789 non-null  object 
 3   MunicipalityCode             184548 non-null  int64  
 4   Prefecture                   184548 non-null  object 
 5   Municipality                 184548 non-null  object 
 6   DistrictName                 184489 non-null  object 
 7   NearestStation               170016 non-null  object 
 8   TimeToNearestStation         169166 non-null  object 
 9   MinTimeToNearestStation      169166 non-null  float64
 10  MaxTimeToNearestStation      167277 non-null  float64
 11  TradePrice                   184548 non-null  int64  
 12  FloorPlan                    31436 non-null   object 
 13 

`TimeToNearestStation` is an object string, while `MinTimeToNearestStation` and `MaxTimeToNearestStation`are either the same time (e.g., the first case in the head() above shows min and max times of 28 minutes) or a approximate range (30-minute min and a 60-min max). These are essentially repeated in `TimeToNearestStation`, making it a somewhat useless column to have.

`Region` is not a geographic location per se, but zone type, such as Residential or Commercial. Approximatey 25% of `Region` data is missing. CityPlanning on the other hand, which also shows zoning categories, is missing only approximately 9% of cases.

In [ ]:
null_summary = pd.DataFrame({
    "null_count": real_estate_df.isnull().sum(),
    "null_percent": round(((real_estate_df.isnull().sum() / len(real_estate_df)) * 100),2)
})

null_summary

,null_count,null_percent
No,0,0.00
Type,0,0.00
Region,46759,25.34
MunicipalityCode,0,0.00
Prefecture,0,0.00
Municipality,0,0.00
DistrictName,59,0.03
NearestStation,14532,7.87
TimeToNearestStation,15382,8.33
MinTimeToNearestStation,15382,8.33


## 3. Data Cleaning and Feature Engineering

The raw MLIT data requires cleaning and feature engineering before it can support downstream analysis. The following steps address data quality, standardization, and the creation of generally useful features:

- **Time to nearest station:** The original field contains text ranges (e.g., `"30-60 Minutes"`). Minimum and maximum travel times are extracted, and average travel time and travel-time range are derived where possible.
- **Building age:** Calculated from transaction year minus building year, where the required information is available.
- **Categorical features:** Categories are standardized or simplified where appropriate. Missing values are retained when their meaning cannot be reliably inferred, rather than automatically replaced with `"Unknown"` or sentinel values.
- **Capped values:** Fields reported as thresholds (e.g., area recorded as 2,000 m² or greater) are accompanied by indicator flags to preserve information about the cap.
- **Property characteristics:** Additional features, such as `HasBuilding` and `AreaSizeCategory`, are created where supported by the source data.

Cleaning and feature engineering are performed before exporting the dataset. Model-specific imputation, feature selection, encoding, and other preprocessing steps are handled in subsequent notebooks.

### 3.1 Station Variables


In [ ]:
diff_exist = real_estate_df["MaxTimeToNearestStation"] != real_estate_df["MinTimeToNearestStation"]
diff_exist.value_counts()

,count
False,132303
True,52245


Although the two variables `MaxTimeToNearestStation` and `MinTimeToNearestStation` appear the same in the head summary earlier, this is only for a little under 25% of them.

In [ ]:
#Check the difference between `MaxTimeToNearestStation` and `MinTimeToNearestStation`

diff = real_estate_df["MaxTimeToNearestStation"] - real_estate_df["MinTimeToNearestStation"]

In [ ]:
diff.describe()

,0
count,167277.000000
mean,6.272351
std,12.199550
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,30.000000


The max difference is 30 minutes, reflecting the 30-60 min range properties. Minimum is 0, showing the cases where MinTimeToNearestStation and MaxTimeToNearestStation are the same. However, the mean and standard deviation show that there are cases where these two distance features are not the same. Therefore, I will engineer two features (AvgTimeToStation and RangeTimeToStation).

In [ ]:
#Engineer AvgTimeToNearestStation

real_estate_df["AvgTimeToNearestStation"] = (
    real_estate_df["MinTimeToNearestStation"] +
    real_estate_df["MaxTimeToNearestStation"]
) / 2

In [ ]:
real_estate_df["AvgTimeToNearestStation"].describe()

,AvgTimeToNearestStation
count,167277.000000
mean,22.746678
std,20.570786
min,0.000000
25%,10.000000
50%,16.000000
75%,26.000000
max,105.000000


In [ ]:
#Engineer TimeRangeToNearestStation

real_estate_df["TimeRangeToNearestStation"] = (
    real_estate_df["MaxTimeToNearestStation"] -
    real_estate_df["MinTimeToNearestStation"]
)

In [ ]:
real_estate_df["TimeRangeToNearestStation"].describe()

,TimeRangeToNearestStation
count,167277.000000
mean,6.272351
std,12.199550
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,30.000000


In [ ]:
# Create mask for AvgTimeToNearestStation > 100
over_100_mask = real_estate_df["AvgTimeToNearestStation"] > 100

# Filter and display the rows
matching_rows = real_estate_df[over_100_mask]
print(f"Total rows found: {len(matching_rows)}")
display(matching_rows[["TimeToNearestStation", "MinTimeToNearestStation", "MaxTimeToNearestStation", "AvgTimeToNearestStation"]].head(10))

Total rows found: 3223


,TimeToNearestStation,MinTimeToNearestStation,MaxTimeToNearestStation,AvgTimeToNearestStation
57,1H30-2H,90.0,120.0,105.0
58,1H30-2H,90.0,120.0,105.0
59,1H30-2H,90.0,120.0,105.0
60,1H30-2H,90.0,120.0,105.0
61,1H30-2H,90.0,120.0,105.0
62,1H30-2H,90.0,120.0,105.0
360,1H30-2H,90.0,120.0,105.0
361,1H30-2H,90.0,120.0,105.0
362,1H30-2H,90.0,120.0,105.0
363,1H30-2H,90.0,120.0,105.0


In [ ]:
#Does station data have a relationship with property type?
real_estate_df.groupby("Type")["AvgTimeToNearestStation"].apply(lambda x: x.isna().mean()).sort_values(ascending=False)

,AvgTimeToNearestStation
Type,
Agricultural Land,1.000000
Forest Land,1.000000
Residential Land(Land Only),0.022639
"Pre-owned Condominiums, etc.",0.020028
Residential Land(Land and Building),0.012424


In [ ]:
# Are there station differeces between types of planning zones?

real_estate_df.groupby("CityPlanning")["AvgTimeToNearestStation"].apply(lambda x: x.isna().mean()).sort_values(ascending=False)

,AvgTimeToNearestStation
CityPlanning,
Quasi-city Planning Area,0.176471
Outside City Planning Area,0.106422
Non-divided City Planning Area,0.050525
Urbanization Control Area,0.030322
Commercial Zone,0.020412
Exclusively Industrial Zone,0.016260
Quasi-industrial Zone,0.013130
Category I Residential Zone,0.012811
Neighborhood Commercial Zone,0.010796


In [ ]:
# When the station name is missing, how often are the average travel time and travel-time range also missing?

real_estate_df[
    real_estate_df["NearestStation"].isna()
][["AvgTimeToNearestStation","TimeRangeToNearestStation"]].isna().mean()

,0
AvgTimeToNearestStation,1.0
TimeRangeToNearestStation,1.0


Null data for station appears to also reflect locations where there is no closeby station, such as outside city planning area or qusai-planning areas.

In [ ]:
# Create HasStation before filling missing travel times
real_estate_df["HasStation"] = (
    real_estate_df["AvgTimeToNearestStation"].notna().astype(int)
)

# Fill missing station names
real_estate_df["NearestStation"] = (
    real_estate_df["NearestStation"].fillna("NoStation")
)

### 3.2 Property Area

`FloorPlan` is over 80% null.

In [ ]:
#Verify Area and AreaIsGreaterFlag

real_estate_df.groupby("AreaIsGreaterFlag")["Area"].describe()

,count,mean,std,min,25%,50%,75%,max
AreaIsGreaterFlag,,,,,,,,
0,180255.0,291.758953,449.833497,10.0,105.0,165.0,250.0,4900.0
1,4293.0,2814.116003,1334.158074,2000.0,2000.0,2000.0,5000.0,5000.0


In [ ]:
#Verify Area and AreaIsGreaterFlag

real_estate_df.groupby("AreaIsGreaterFlag")["Area"].count()

,Area
AreaIsGreaterFlag,
0,180255
1,4293


In [ ]:
# Filter for AreaIsGreaterFlag == 1 and count the distribution of Area values
flagged_areas = real_estate_df[real_estate_df['AreaIsGreaterFlag'] == 1]['Area']
counts = flagged_areas.value_counts(dropna=False)

print("=== Distribution of Area when AreaIsGreaterFlag is 1 ===")
display(counts)

=== Distribution of Area when AreaIsGreaterFlag is 1 ===


,count
Area,
2000,3128
5000,1165


`AreaIsGreaterFlag` shows us that flags are properties greater than 2000 or 5000 squre meters. It does not tell us what the area actually is.

In [ ]:
real_estate_df["AreaSizeCategory"] = np.select(
    [
        real_estate_df["Area"] == 2000,
        real_estate_df["Area"] == 5000
    ],
    [
        "2000-4999",
        "4999 or more"
    ],
    default="1999 or less"
)

In [ ]:
real_estate_df.groupby("AreaSizeCategory").mean(numeric_only=True)

,No,MunicipalityCode,MinTimeToNearestStation,MaxTimeToNearestStation,TradePrice,Area,AreaIsGreaterFlag,UnitPrice,PricePerTsubo,Frontage,FrontageIsGreaterFlag,TotalFloorArea,TotalFloorAreaIsGreaterFlag,BuildingYear,PrewarBuilding,Breadth,CoverageRatio,FloorAreaRatio,Year,Quarter,AvgTimeToNearestStation,TimeRangeToNearestStation,HasStation
AreaSizeCategory,,,,,,,,,,,,,,,,,,,,,,,
1999 or less,91768.221512,12207.006988,20.484327,25.551420,2.242942e+07,288.216775,0.000000,77638.995407,256680.870850,13.243647,0.006543,134.311103,0.000784,1997.143672,0.000656,6.253256,57.761177,173.380741,2012.896610,2.505109,22.487295,6.128251,0.913393
2000-4999,107015.403599,12224.497572,33.925104,44.193342,2.228922e+08,2000.000000,0.893459,36241.737106,119956.284801,42.963225,0.413596,1144.459302,0.079977,1988.350540,0.001143,9.288679,59.086925,186.837107,2012.902599,2.441017,37.076664,14.233356,0.849472
4999 or more,126147.876395,12244.345923,NaN,NaN,1.887577e+07,5000.000000,1.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,NaN,2012.988841,2.466094,NaN,NaN,0.000000


One thing stands out here: Property over 4999 squared meters are missing a lot of data.

In [ ]:
cols_with_nan = [
    "MinTimeToNearestStation",
    "MaxTimeToNearestStation",
    "UnitPrice",
    "PricePerTsubo",
    "Frontage",
    "TotalFloorArea",
    "BuildingYear",
    "Breadth",
    "CoverageRatio",
    "FloorAreaRatio",
    "AvgTimeToNearestStation"
]

grouped = real_estate_df.groupby("AreaSizeCategory")

# Total observations per category (once)
total = grouped.size().rename("Total")

# Percentage of non-null values for each column
nonnull_pct = pd.concat(
    {
        col: grouped[col].apply(lambda x: x.notna().mean() * 100)
        for col in cols_with_nan
    },
    axis=1
).rename(columns=lambda col: f"{col} NonNull (%)")

# Combine into one table
counts = pd.concat([total, nonnull_pct], axis=1).round(2)

counts

,Total,MinTimeToNearestStation NonNull (%),MaxTimeToNearestStation NonNull (%),UnitPrice NonNull (%),PricePerTsubo NonNull (%),Frontage NonNull (%),TotalFloorArea NonNull (%),BuildingYear NonNull (%),Breadth NonNull (%),CoverageRatio NonNull (%),FloorAreaRatio NonNull (%),AvgTimeToNearestStation NonNull (%)
AreaSizeCategory,,,,,,,,,,,,
1999 or less,179882,92.31,91.34,31.71,31.71,66.55,41.10,58.10,72.52,87.44,87.44,91.34
2000-4999,3501,88.86,84.95,62.58,62.58,73.86,24.56,23.79,83.00,78.21,78.21,84.95
4999 or more,1165,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Smaller properties (less than 2000 sq. meters) are mising price data for approx. 70% of cases. 67.8% of cases total (regardless of size) are missing price data. As we go through the categories, determining if there are particular observation categories that do not record price data should be inspected.

### 3.3 Unit Price and Price per Tsubo

In [ ]:

real_estate_df.groupby("Type")["UnitPrice"].apply(
    lambda x: x.isna().mean()
).sort_values(ascending=False)

,UnitPrice
Type,
Agricultural Land,1.0
Forest Land,1.0
"Pre-owned Condominiums, etc.",1.0
Residential Land(Land and Building),1.0
Residential Land(Land Only),0.0


In [ ]:
real_estate_df.groupby("Type")["PricePerTsubo"].apply(
    lambda x: x.isna().mean()
).sort_values(ascending=False)

,PricePerTsubo
Type,
Agricultural Land,1.0
Forest Land,1.0
"Pre-owned Condominiums, etc.",1.0
Residential Land(Land and Building),1.0
Residential Land(Land Only),0.0


`UnitPrice` and `PricePerTsubo` are used only for Residental Land (Land Only)

### 3.4 Landshape and Frontage

In [ ]:
#Inspect Landshape

real_estate_df["LandShape"].unique()

array(['Rectangular Shaped', 'Irregular Shaped', nan,
       'Semi-rectangular Shaped', 'Semi-square Shaped',
       'Trapezoidal Shaped', 'Semi-trapezoidal Shaped', 'Semi-shaped',
       'Square Shaped', 'Flag-shaped etc.'], dtype=object)

In [ ]:
#Replace NaN in Landshape with "Unknown"

real_estate_df["LandShape"] = real_estate_df["LandShape"].fillna("Unknown")
real_estate_df["LandShape"].isnull().sum()

np.int64(0)

In [ ]:
#Reduce noise by combining cases of similar shape

real_estate_df["LandShapeReplaced"] = real_estate_df["LandShape"].replace({
    "Semi-square Shaped": "Square",
    "Semi-rectangular Shaped": "Rectangular",
    "Semi-trapezoidal Shaped": "Trapezoidal"
})

In [ ]:
#Inspect Frontage Flag
real_estate_df.groupby("FrontageIsGreaterFlag")["Frontage"].describe()

,count,mean,std,min,25%,50%,75%,max
FrontageIsGreaterFlag,,,,,,,,
False,119668.0,13.079604,7.116671,0.5,9.0,11.6,15.0,49.9
True,2625.0,50.000000,0.000000,50.0,50.0,50.0,50.0,50.0


Flag caps and censors frontage at 50m.

In [ ]:
#Replace the 35% missing values with -1 to allow later modeling

real_estate_df["FrontageReplaced"] = real_estate_df["Frontage"].fillna(-1)
real_estate_df["FrontageReplaced"].isnull().sum()

np.int64(0)

### 3.5 Floor Area

In [ ]:
#Inspect Total Floor Area Flag

real_estate_df.groupby("TotalFloorAreaIsGreaterFlag")["TotalFloorArea"].describe()

,count,mean,std,min,25%,50%,75%,max
TotalFloorAreaIsGreaterFlag,,,,,,,,
0,74376.0,135.430717,147.884653,10.0,95.0,105.0,120.0,1900.0
1,421.0,2000.000000,0.000000,2000.0,2000.0,2000.0,2000.0,2000.0


Flag caps and censors floor area at 2000 sq.meters.

In [ ]:
#Inspect if Land Only purchases influence Floor Area feature

real_estate_df["TotalFloorArea"].isnull().groupby(real_estate_df["Type"]).mean()

,TotalFloorArea
Type,
Agricultural Land,1.000000
Forest Land,1.000000
"Pre-owned Condominiums, etc.",1.000000
Residential Land(Land Only),1.000000
Residential Land(Land and Building),0.047851


Land-only transactions are 100% null:  expected because there may be no building.

Agricultural and forest land are 100% null: also expected because these transactions concern land rather than building space.

Pre-owned condominiums: This is the interesting exception. A condominium normally has a floor area, so 100% missingness warrants further investigation.

Most residential properties with a building have floor area recorded; only 4.79% cases are missing floor area.

In [ ]:
# Compare feature distributions by property type
land_types = ["Residential Land(Land and Building)", "Residential Land(Land Only)","Agricultural Land", "Forest Land", "Pre-owned Condominiums, etc."]

real_estate_df.loc[
    real_estate_df["Type"].isin(land_types)
].groupby("Type").agg(
    Rows=("Type", "size"),
    TotalFloorArea_NonNull=("TotalFloorArea", "count"),
    BuildingYear_NonNull=("BuildingYear", "count"),
    Frontage_NonNull=("Frontage", "count"),
    CoverageRatio_NonNull=("CoverageRatio", "count"),
    TradePrice_NonNull=("TradePrice", "count"),
    Area_NonNull=("Area", "count"),
    UnitPrice_NonNull=("UnitPrice", "count"),
    PricePerTsubo_NonNull=("PricePerTsubo", "count"),
    Region_NonNull=("Region", "count"),
    CityPlanning_NonNull=("CityPlanning", "count"),
    Use_NonNull=("Use", "count"),
   )

,Rows,TotalFloorArea_NonNull,BuildingYear_NonNull,Frontage_NonNull,CoverageRatio_NonNull,TradePrice_NonNull,Area_NonNull,UnitPrice_NonNull,PricePerTsubo_NonNull,Region_NonNull,CityPlanning_NonNull,Use_NonNull
Type,,,,,,,,,,,,
Agricultural Land,9300,0,0,0,0,9300,9300,0,0,0,0,0
Forest Land,5004,0,0,0,0,5004,5004,0,0,0,0,0
"Pre-owned Condominiums, etc.",32455,0,31251,0,30935,32455,32455,0,0,0,31170,30136
Residential Land(Land Only),59233,0,0,51967,54515,59233,59233,59233,59233,59233,58843,0
Residential Land(Land and Building),78556,74797,74098,70326,74576,78556,78556,0,0,78556,78170,73757


`UnitPrice` and PricePerTsubo are recorded for Residential Land only.

`BuildingYear` is recorded for Residential Buildings and Pre-owned Condos; Land only transactions may have a building on site (barn, etc.), but these are not recorded or considered.

`TotalFloorArea` is not recorded for Pre-owned condos

`Region` (zoning) is recorded for residential land categories only, but `CityPlanning` also includes Pre-owned condos.

`Use` is recorded for Residential buildings and Pre-owned condos

In [ ]:
#Create residential building indicator feature
#This intentionally ignores possible agricultural buidlings
building_map = {
    "Agricultural Land":0,
    "Forest Land":0,
    "Residential Land(Land Only)": 0,
    "Residential Land(Land and Building)": 1,
    "Pre-owned Condominiums, etc.": 1
}

real_estate_df["HasBuilding"] = real_estate_df["Type"].map(building_map)

In [ ]:
# Land-only properties with null values for TotalFloorArea have Na replaced with 0

mask = real_estate_df["HasBuilding"] == 0

real_estate_df.loc[mask, "TotalFloorArea"] = real_estate_df.loc[mask, "TotalFloorArea"].fillna(0)

### 3.6 Building Year

In [ ]:
#Inspect missing values in BuildingYear
real_estate_df["BuildingYear"].isnull().groupby(real_estate_df["HasBuilding"]).mean()

,BuildingYear
HasBuilding,
0,1.000000
1,0.051004


In [ ]:
real_estate_df.groupby("HasBuilding")["BuildingYear"].agg(
    TotalRows="size",
    MissingCount=lambda s: s.isna().sum(),
    AvailableCount=lambda s: s.notna().sum()
)

,TotalRows,MissingCount,AvailableCount
HasBuilding,,,
0,73537,73537,0
1,111011,5662,105349


Approx 5% of observations of properites with buildings are missing `BuildingYear`

In [ ]:
real_estate_df["BuildingYear"].dropna().sort_values().unique().tolist()

[1945.0,
 1946.0,
 1947.0,
 1948.0,
 1949.0,
 1950.0,
 1951.0,
 1952.0,
 1953.0,
 1954.0,
 1955.0,
 1956.0,
 1957.0,
 1958.0,
 1959.0,
 1960.0,
 1961.0,
 1962.0,
 1963.0,
 1964.0,
 1965.0,
 1966.0,
 1967.0,
 1968.0,
 1969.0,
 1970.0,
 1971.0,
 1972.0,
 1973.0,
 1974.0,
 1975.0,
 1976.0,
 1977.0,
 1978.0,
 1979.0,
 1980.0,
 1981.0,
 1982.0,
 1983.0,
 1984.0,
 1985.0,
 1986.0,
 1987.0,
 1988.0,
 1989.0,
 1990.0,
 1991.0,
 1992.0,
 1993.0,
 1994.0,
 1995.0,
 1996.0,
 1997.0,
 1998.0,
 1999.0,
 2000.0,
 2001.0,
 2002.0,
 2003.0,
 2004.0,
 2005.0,
 2006.0,
 2007.0,
 2008.0,
 2009.0,
 2010.0,
 2011.0,
 2012.0,
 2013.0,
 2014.0,
 2015.0,
 2016.0,
 2017.0,
 2018.0,
 2019.0,
 2020.0]

Dates range from 1945 to 2020, excluding any null values (approx 5% of cases)

In [ ]:
real_estate_df["BuildingYear"].isnull().groupby(real_estate_df["PrewarBuilding"]).mean()

,BuildingYear
PrewarBuilding,
0,0.429435
1,0.000000


In [ ]:
mask = real_estate_df["PrewarBuilding"] == 1

real_estate_df.loc[mask, "BuildingYear"].describe()

,BuildingYear
count,122.0
mean,1945.0
std,0.0
min,1945.0
25%,1945.0
50%,1945.0
75%,1945.0
max,1945.0


In [ ]:
# Do null values in BuildingYear have no building?

mask = real_estate_df["HasBuilding"] == 0

real_estate_df.loc[mask, "BuildingYear"].count()


np.int64(0)

All prewar buildings (N = 122) have a 1945 placeholder for their `BuildingYear`

None of the land-only rows have a recorded building year.

In [ ]:
#Assign year to prewar buildings

#real_estate_df["BuildingYearReplaced"] = real_estate_df["BuildingYear"]

#real_estate_df.loc[
   # real_estate_df["PrewarBuilding"] == 1,
    #"BuildingYearReplaced"
#] = 1944

In [ ]:
#Engineer Building Age

real_estate_df["BuildingAge"] = real_estate_df["Year"] - real_estate_df["BuildingYear"]
real_estate_df["BuildingAge"].head(20)

,BuildingAge
0,NaN
1,NaN
2,NaN
3,21.0
4,10.0
5,19.0
6,NaN
7,NaN
8,NaN
9,17.0


In [ ]:
#Inspect properties with buildings

mask = real_estate_df["HasBuilding"] == 1

real_estate_df.loc[mask, "BuildingAge"].describe()

,BuildingAge
count,105349.000000
mean,15.914949
std,14.940407
min,-2.000000
25%,0.000000
50%,14.000000
75%,28.000000
max,74.000000


The minimum is a negative number, which is impossible, warranting further inspection.

In [ ]:
mask = real_estate_df["BuildingAge"] == -2
real_estate_df.loc[mask].head(20)

,No,Type,Region,MunicipalityCode,Prefecture,Municipality,DistrictName,NearestStation,TimeToNearestStation,MinTimeToNearestStation,MaxTimeToNearestStation,TradePrice,FloorPlan,Area,AreaIsGreaterFlag,UnitPrice,PricePerTsubo,LandShape,Frontage,FrontageIsGreaterFlag,TotalFloorArea,TotalFloorAreaIsGreaterFlag,BuildingYear,PrewarBuilding,Structure,Use,Purpose,Direction,Classification,Breadth,CityPlanning,CoverageRatio,FloorAreaRatio,Period,Year,Quarter,Renovation,Remarks,AvgTimeToNearestStation,TimeRangeToNearestStation,HasStation,AreaSizeCategory,LandShapeReplaced,FrontageReplaced,HasBuilding,BuildingAge
33404,33405,Residential Land(Land and Building),Commercial Area,12101,Chiba Prefecture,"Chuo Ward,Chiba City",Shinjuku,Chiba,11,11.0,11.0,2100000000,NaN,1500,0,NaN,NaN,Irregular Shaped,20.0,False,2000.0,1,2008.0,0,RC,Housing Complex,NaN,Southeast,City Road,15.0,Neighborhood Commercial Zone,80.0,400.0,4th quarter 2006,2006,4,NaN,NaN,11.0,0.0,1,1999 or less,Irregular Shaped,20.0,1,-2.0
98643,98644,Residential Land(Land and Building),Residential Area,12207,Chiba Prefecture,Matsudo City,Kamihongo,Kitamatsudo,11,11.0,11.0,41000000,NaN,130,0,NaN,NaN,Rectangular Shaped,7.0,False,95.0,0,2011.0,0,W,House,NaN,Southeast,City Road,6.0,Category I Exclusively Medium-high Residential...,60.0,200.0,1st quarter 2009,2009,1,NaN,NaN,11.0,0.0,1,1999 or less,Rectangular Shaped,7.0,1,-2.0
133591,133592,Residential Land(Land and Building),Residential Area,12231,Chiba Prefecture,Inzai City,Makinohara,Inzaimakinohara,21,21.0,21.0,44000000,NaN,170,0,NaN,NaN,Semi-rectangular Shaped,10.0,False,110.0,0,2019.0,0,W,House,House,Northwest,Road,6.0,Category I Exclusively Low-story Residential Zone,40.0,80.0,4th quarter 2017,2017,4,NaN,NaN,21.0,0.0,1,1999 or less,Rectangular,10.0,1,-2.0


Case 1 is a classic pre-construction commercial sale. A developer or institutional investor contracted a multi-billion yen reinforced concrete housing complex project in 2006, which took two years to construct and officially open in 2008.

Cases 2 and 3 are standard wooden residential houses. While it is possible to buy custom homes off-plan, wooden residential builds rarely take a full 2 years to complete once a consumer transaction is registered. This pattern is usually caused by delayed registry logging or survey data adjustments (e.g., the land contract transaction was completed and reported, but the official structural completion certificate wasn't filed/processed until two years later).
The market pricing is based on a "brand-new build" evaluation, meaning the physical depreciation is effectively zero at the point of contract.

For all 3 cases, `BuildingAge` will be clipped to 0.

In [ ]:
#Clip BuildingAge to 0

real_estate_df["BuildingAge"] = real_estate_df["BuildingAge"].clip(lower=0)

mask = real_estate_df["HasBuilding"] == 1
real_estate_df.loc[mask, "BuildingAge"].describe()

,BuildingAge
count,105349.000000
mean,15.936316
std,14.916895
min,0.000000
25%,0.000000
50%,14.000000
75%,28.000000
max,74.000000


In [ ]:
# Replace BuildingAge with -1 for properties without a building
mask = real_estate_df["HasBuilding"] == 0

real_estate_df.loc[mask, "BuildingAge"] = -1

# Inspect the first 10 rows
real_estate_df[["Type","BuildingAge"]].head(10)

,Type,BuildingAge
0,Residential Land(Land Only),-1.0
1,Residential Land(Land Only),-1.0
2,Residential Land(Land Only),-1.0
3,"Pre-owned Condominiums, etc.",21.0
4,Residential Land(Land and Building),10.0
5,Residential Land(Land and Building),19.0
6,Residential Land(Land Only),-1.0
7,Residential Land(Land and Building),NaN
8,Residential Land(Land Only),-1.0
9,"Pre-owned Condominiums, etc.",17.0


### 3.7 Structure

In [ ]:
real_estate_df["Structure"].value_counts(dropna=False)

,count
Structure,
NaN,78367
W,63704
RC,26357
SRC,7436
S,4365
LS,4113
B,62
"S, W",51
"RC, W",51


In [ ]:
real_estate_df["Structure"].isnull().groupby(real_estate_df["HasBuilding"]).mean()

,Structure
HasBuilding,
0,1.000000
1,0.043509


In [ ]:
pd.crosstab(
    real_estate_df["Type"],
    real_estate_df["Structure"],
    margins=True,
    dropna=False
)

Structure,B,"B, LS",LS,RC,"RC, B","RC, LS","RC, S","RC, W",S,"S, LS","S, W","S, W, LS",SRC,"SRC, RC",W,"W, B","W, LS",NaN,All
Type,,,,,,,,,,,,,,,,,,,
Agricultural Land,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,9300,9300
Forest Land,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5004,5004
"Pre-owned Condominiums, etc.",0,0,0,24501,0,0,0,0,120,0,0,0,7209,5,16,0,0,604,32455
Residential Land(Land Only),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,59233,59233
Residential Land(Land and Building),62,1,4113,1856,3,3,5,51,4245,5,51,1,227,0,63688,3,16,4226,78556
All,62,1,4113,26357,3,3,5,51,4365,5,51,1,7436,5,63704,3,16,0,184548


In [ ]:
#Structure has many mulitple rows. Will only keep first category

real_estate_df["StructureCleaned"] = real_estate_df["Structure"].str.split(",").str[0]
real_estate_df["StructureCleaned"].unique()

array([nan, 'RC', 'W', 'LS', 'S', 'SRC', 'B'], dtype=object)

In [ ]:
mask = real_estate_df["HasBuilding"] == 0

real_estate_df["StructureCleanedReplaced"] = real_estate_df["StructureCleaned"]
real_estate_df.loc[mask, "StructureCleanedReplaced"] = (
    real_estate_df.loc[mask, "StructureCleaned"].fillna("No Building")
)

real_estate_df["StructureCleanedReplaced"].unique()

array(['No Building', 'RC', 'W', nan, 'LS', 'S', 'SRC', 'B'], dtype=object)

### 3.8 Use and Purpose

In [ ]:
real_estate_df["Use"].value_counts(dropna=False)

,count
Use,
House,93625
NaN,80655
Housing Complex,4278
"House, Shop",965
Shop,714
...,...
"Office, Workshop, Parking Lot",1
"House, Office, Workshop, Warehouse, Shop",1
"Warehouse, Parking Lot",1


In [ ]:
real_estate_df["Use"].isnull().groupby(real_estate_df["HasBuilding"]).mean()

,Use
HasBuilding,
0,1.00000
1,0.06412


In [ ]:
real_estate_df["Purpose"].isnull().groupby(real_estate_df["HasBuilding"]).mean()

,Purpose
HasBuilding,
0,0.801923
1,0.590833


The smaller number of categories in `Purpose` make it more useful for modeling (`Use` would probably cause over correction). However, missingness is a downside for `Purpose`. We can potentially impute missing Purpose with `Use`.

In [ ]:
#Cross-reference purpose and use

pd.set_option("display.max_rows", None)

pd.crosstab(
    real_estate_df["Use"],
    real_estate_df["Purpose"],
    margins=True,
    dropna=False
)

Purpose,Factory,House,Office,Other,Shop,Warehouse,NaN,All
Use,,,,,,,,
Factory,40,2,2,9,2,15,167,237.0
"Factory, Office",8,2,10,2,1,7,89,119.0
"Factory, Office, Other",0,0,1,0,0,0,3,4.0
"Factory, Office, Parking Lot",1,0,0,0,0,0,1,2.0
"Factory, Office, Shop",0,0,0,0,1,0,2,3.0
"Factory, Office, Warehouse",5,0,1,0,0,3,22,31.0
"Factory, Office, Warehouse, Other",0,0,0,0,0,0,1,1.0
"Factory, Office, Workshop",1,0,1,0,0,0,1,3.0
"Factory, Other",0,0,0,0,0,1,2,3.0


In [ ]:
#Use has many mulitple rows. Will only keep first category

real_estate_df["UseSimple"] = real_estate_df["Use"].str.split(",").str[0]
real_estate_df["UseSimple"].unique()

array([nan, 'House', 'Housing Complex', 'Office', 'Other', 'Shop',
       'Warehouse', 'Parking Lot', 'Factory', 'Workshop'], dtype=object)

In [ ]:
#Redo Crosstab with a simplified Use feature

pd.crosstab(
    real_estate_df["UseSimple"],
    real_estate_df["Purpose"],
    margins=True,
    dropna=False
)

Purpose,Factory,House,Office,Other,Shop,Warehouse,NaN,All
UseSimple,,,,,,,,
Factory,59,4,17,13,6,28,322,449.0
House,14,35669,273,1619,173,123,57727,95598.0
Housing Complex,1,1834,25,443,28,12,2649,4992.0
Office,10,39,178,65,71,46,801,1210.0
Other,1,14,11,57,37,3,268,391.0
Parking Lot,0,3,0,7,3,3,31,47.0
Shop,2,24,26,45,172,8,462,739.0
Warehouse,16,11,8,24,9,63,244,375.0
Workshop,7,3,1,6,2,7,66,92.0


In [ ]:

# Calculate the dominant-purpose rate among known values

purpose_counts = pd.crosstab(
    real_estate_df["UseSimple"],
    real_estate_df["Purpose"]
)

known_purpose_counts = purpose_counts.drop(
    columns=[col for col in purpose_counts.columns if pd.isna(col)],
    errors="ignore"
)

summary = pd.DataFrame({
    "KnownPurposeCount": known_purpose_counts.sum(axis=1),
    "MostCommonPurpose": known_purpose_counts.idxmax(axis=1),
    "DominantPurposeCount": known_purpose_counts.max(axis=1),
    "DominantPurposePct": (
        known_purpose_counts.max(axis=1)
        / known_purpose_counts.sum(axis=1)
        * 100
    )
}).sort_values("KnownPurposeCount", ascending=False)

summary.round(2)

,KnownPurposeCount,MostCommonPurpose,DominantPurposeCount,DominantPurposePct
UseSimple,,,,
House,37871,House,35669,94.19
Housing Complex,2343,House,1834,78.28
Office,409,Office,178,43.52
Shop,277,Shop,172,62.09
Warehouse,131,Warehouse,63,48.09
Factory,127,Factory,59,46.46
Other,123,Other,57,46.34
Workshop,26,Factory,7,26.92
Parking Lot,16,Other,7,43.75


In [ ]:
# Preserve the original Purpose column
real_estate_df["PurposeImputed"] = real_estate_df["Purpose"].copy()

# Reproducible random sampling
rng = np.random.default_rng(42)

# Rows where Purpose is missing and UseSimple is present
mask = (
    real_estate_df["PurposeImputed"].isna()
    & real_estate_df["UseSimple"].notna()
)

# Impute from the observed Purpose distribution within each UseSimple category
for use_value, idx in real_estate_df.loc[mask].groupby("UseSimple").groups.items():
    observed = real_estate_df.loc[
        (real_estate_df["UseSimple"] == use_value)
        & real_estate_df["Purpose"].notna(),
        "Purpose"
    ]

    if not observed.empty:
        probabilities = observed.value_counts(normalize=True)

        real_estate_df.loc[idx, "PurposeImputed"] = rng.choice(
            probabilities.index,
            size=len(idx),
            p=probabilities.values
        )

In [ ]:
#Check imputed Purpose data

pd.crosstab(
    real_estate_df["UseSimple"],
    real_estate_df["PurposeImputed"],
    margins=True,
    dropna=False
)

PurposeImputed,Factory,House,Office,Other,Shop,Warehouse,NaN,All
UseSimple,,,,,,,,
Factory,215,10,60,44,22,98,0,449.0
House,34,90029,678,4098,444,315,0,95598.0
Housing Complex,2,3896,57,946,59,32,0,4992.0
Office,26,105,518,198,216,147,0,1210.0
Other,1,43,35,190,115,7,0,391.0
Parking Lot,0,11,0,19,9,8,0,47.0
Shop,3,67,61,130,457,21,0,739.0
Warehouse,51,27,20,79,32,166,0,375.0
Workshop,28,15,2,22,5,20,0,92.0


### 3.9 Direction (Facing the Sun)

In [ ]:
#Inspect Direction

real_estate_df.groupby("Type")["Direction"].value_counts(dropna=False)

Type                                 Direction     
Agricultural Land                    NaN                9300
Forest Land                          NaN                5004
Pre-owned Condominiums, etc.         NaN               32455
Residential Land(Land Only)          Southeast          8905
                                     Southwest          8357
                                     Northwest          7884
                                     Northeast          7443
                                     South              7069
                                     North              6113
                                     East               5690
                                     West               5447
                                     No facing road     1892
                                     NaN                 433
Residential Land(Land and Building)  Southeast         12355
                                     Southwest         11555
                                     Northeast         11339
                                     Northwest         11051
                                     South              9448
                                     North              7970
                                     East               7217
                                     West               6997
                                     NaN                 381
                                     No facing road      243
Name: count, dtype: int64

For agricultural land, forest land, and pre-owned condominiums, `Direction` is always missing in this dataset.

For the two residential land categories, missingness is very low.

This suggests that `Direction` is largely not applicable or not recorded for certain property types, rather than being randomly missing across the dataset.

### 3.10 Road Classification

In [ ]:
#Inspect Classification

real_estate_df["Classification"].value_counts(dropna=False)

,count
Classification,
City Road,91183
NaN,49708
Private Road,24484
Road,5768
Prefectural Road,4621
Town Road,4174
National Highway,2117
Access Road,1677
Village Road,423


In [ ]:
real_estate_df.groupby("Type")["Classification"].value_counts(dropna=False)

Type                                 Classification               
Agricultural Land                    NaN                               9300
Forest Land                          NaN                               5004
Pre-owned Condominiums, etc.         NaN                              32455
Residential Land(Land Only)          City Road                        37581
                                     Private Road                      9279
                                     Road                              2682
                                     Prefectural Road                  2580
                                     NaN                               2325
                                     Town Road                         2236
                                     National Highway                  1204
                                     Access Road                        885
                                     Village Road                       198
                                     Agricultural Road                  159
                                     Ward Road                           42
                                     Forest Road                         28
                                     Kyoto/ Osaka Prefectural Road       22
                                     Hokkaido Prefectural Road           12
Residential Land(Land and Building)  City Road                        53602
                                     Private Road                     15205
                                     Road                              3086
                                     Prefectural Road                  2041
                                     Town Road                         1938
                                     National Highway                   913
                                     Access Road                        792
                                     NaN                                624
                                     Village Road                       225
                                     Ward Road                           50
                                     Forest Road                         26
                                     Agricultural Road                   22
                                     Hokkaido Prefectural Road           18
                                     Kyoto/ Osaka Prefectural Road       13
                                     Tokyo Metropolitan Road              1
Name: count, dtype: int64

Similar to other features, Agricultural Land, Forest Land, and Pre-owned condos do not have Classification recorded; this accounts for 46,759 null entries (approx. 94% of all null entries).

### 3.11 Frontage Road Breadth (meters)

In [ ]:
#Inspect Breadth
real_estate_df["Breadth"].describe()

,Breadth
count,133357.000000
mean,6.319401
std,3.701414
min,1.000000
25%,4.200000
50%,6.000000
75%,6.000000
max,80.000000


In [ ]:
real_estate_df.groupby("Type")["Breadth"].mean()

,Breadth
Type,
Agricultural Land,NaN
Forest Land,NaN
"Pre-owned Condominiums, etc.",NaN
Residential Land(Land Only),6.492498
Residential Land(Land and Building),6.192921


Similar to other features, Agricultural Land, Forest Land, and Pre-owned condos do not have Breadth recorded.

## 3.12 City Planning (Zoning)

In [ ]:
#Inspect CityPlanning

real_estate_df["CityPlanning"].value_counts(dropna=False)

,count
CityPlanning,
Category I Exclusively Low-story Residential Zone,55435
Category I Residential Zone,30677
Category I Exclusively Medium-high Residential Zone,25766
NaN,16365
Urbanization Control Area,14709
Non-divided City Planning Area,12390
Commercial Zone,5144
Neighborhood Commercial Zone,4446
Outside City Planning Area,4407


In [ ]:
#Check imputed Purpose data

pd.crosstab(
    real_estate_df["Type"],
    real_estate_df["CityPlanning"],
    margins=True,
    dropna=False
)

CityPlanning,Category I Exclusively Low-story Residential Zone,Category I Exclusively Medium-high Residential Zone,Category I Residential Zone,Category II Exclusively Low-story Residential Zone,Category II Exclusively Medium-high Residential Zone,Category II Residential Zone,Commercial Zone,Exclusively Industrial Zone,Industrial Zone,Neighborhood Commercial Zone,Non-divided City Planning Area,Outside City Planning Area,Quasi-city Planning Area,Quasi-industrial Zone,Quasi-residential Zone,Urbanization Control Area,NaN,All
Type,,,,,,,,,,,,,,,,,,
Agricultural Land,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,9300,9300
Forest Land,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5004,5004
"Pre-owned Condominiums, etc.",1862,11277,7312,56,1106,1183,3508,14,850,1729,8,7,0,1477,632,149,1285,32455
Residential Land(Land Only),19598,5498,10412,216,611,1384,762,187,444,1398,6808,2474,12,994,433,7612,390,59233
Residential Land(Land and Building),33975,8991,12953,281,1234,1568,874,168,681,1319,5574,1926,5,1261,412,6948,386,78556
All,55435,25766,30677,553,2951,4135,5144,369,1975,4446,12390,4407,17,3732,1477,14709,0,184548


City Planning is null for Agicultural and Forest Land. It has been recorded for Pre-owned Condos; approx 4% of pre-owned condo observations are missing this data.

In [ ]:
#Replace null data for agricultural and forest land

# Preserve the original column
real_estate_df["CityPlanningCleaned"] = real_estate_df["CityPlanning"]

# Replace nulls based on property type
real_estate_df.loc[
    real_estate_df["Type"] == "Agricultural Land", "CityPlanningCleaned"
] = real_estate_df.loc[
    real_estate_df["Type"] == "Agricultural Land", "CityPlanningCleaned"
].fillna("Agricultural Land")

real_estate_df.loc[
    real_estate_df["Type"] == "Forest Land", "CityPlanningCleaned"
] = real_estate_df.loc[
    real_estate_df["Type"] == "Forest Land", "CityPlanningCleaned"
].fillna("Forest Land")

In [ ]:
real_estate_df.groupby("Type")["CityPlanningCleaned"].apply(
    lambda s: s.isna().sum()
)

,CityPlanningCleaned
Type,
Agricultural Land,0
Forest Land,0
"Pre-owned Condominiums, etc.",1285
Residential Land(Land Only),390
Residential Land(Land and Building),386


In [ ]:
# Recheck the relationship between station distance and planning zones

real_estate_df.groupby("CityPlanningCleaned")["AvgTimeToNearestStation"].apply(lambda x: x.isna().mean()).sort_values(ascending=False)

,AvgTimeToNearestStation
CityPlanningCleaned,
Agricultural Land,1.000000
Forest Land,1.000000
Quasi-city Planning Area,0.176471
Outside City Planning Area,0.106422
Non-divided City Planning Area,0.050525
Urbanization Control Area,0.030322
Commercial Zone,0.020412
Exclusively Industrial Zone,0.016260
Quasi-industrial Zone,0.013130


Above strengthen the argument that null data for stations is likely due to no station being near to the property.

### 3.13 Coverage Ratio

(Maximum Building Coverage Ratio(%), i.e., the percentage of a plot of land that can be covered by a building’s footprint)

In [ ]:
#Inspect Coverage Ratio

real_estate_df["CoverageRatio"].value_counts(dropna=False)

,count
CoverageRatio,
60.0,98831
50.0,45270
NaN,24522
80.0,9492
40.0,4157
70.0,1491
30.0,785


In [ ]:
real_estate_df.groupby("Type")["CoverageRatio"].value_counts(dropna=False)

Type                                 CoverageRatio
Agricultural Land                    NaN               9300
Forest Land                          NaN               5004
Pre-owned Condominiums, etc.         60.0             24318
                                     80.0              5164
                                     NaN               1520
                                     50.0              1219
                                     40.0               132
                                     30.0               101
                                     70.0                 1
Residential Land(Land Only)          60.0             33539
                                     50.0             16041
                                     NaN               4718
                                     80.0              2147
                                     40.0              1548
                                     70.0               841
                                     30.0               399
Residential Land(Land and Building)  60.0             40974
                                     50.0             28010
                                     NaN               3980
                                     40.0              2477
                                     80.0              2181
                                     70.0               649
                                     30.0               285
Name: count, dtype: int64

Agricultural and Forest land do not have coverage ratios because they do not have buildings covering them.

### 3.14 Floor Area Ratio
(maximum legally permitted total floor area of a building, expressed as a percentage of the land area)

In [ ]:
#Inspect FloorAreaRatio

real_estate_df["FloorAreaRatio"].value_counts(dropna=False)

,count
FloorAreaRatio,
200.0,92696
100.0,45748
NaN,24522
150.0,9367
400.0,4363
80.0,4033
300.0,1840
50.0,666
600.0,622


In [ ]:
real_estate_df.groupby("Type")["FloorAreaRatio"].value_counts(dropna=False)

Type                                 FloorAreaRatio
Agricultural Land                    NaN                9300
Forest Land                          NaN                5004
Pre-owned Condominiums, etc.         200.0             24231
                                     400.0              2777
                                     NaN                1520
                                     100.0              1218
                                     300.0              1020
                                     150.0               730
                                     600.0               479
                                     500.0               297
                                     80.0                133
                                     800.0                40
                                     60.0                  8
                                     700.0                 1
                                     900.0                 1
Residential Land(Land Only)          200.0             31670
                                     100.0             16327
                                     NaN                4718
                                     150.0              3302
                                     80.0               1537
                                     400.0               762
                                     50.0                388
                                     300.0               384
                                     60.0                 64
                                     600.0                53
                                     500.0                22
                                     800.0                 6
Residential Land(Land and Building)  200.0             36795
                                     100.0             28203
                                     150.0              5335
                                     NaN                3980
                                     80.0               2363
                                     400.0               824
                                     300.0               436
                                     50.0                278
                                     60.0                211
                                     600.0                90
                                     500.0                36
                                     800.0                 5
Name: count, dtype: int64

In [ ]:
coverage_missing = real_estate_df["CoverageRatio"].isna()
floor_missing = real_estate_df["FloorAreaRatio"].isna()

print("Same missing rows:", coverage_missing.equals(floor_missing))
print("Different missing rows:", (coverage_missing != floor_missing).sum())

Same missing rows: True
Different missing rows: 0


CoverageRatio and FloorAreaRatio are missing identically, including for building properties.

### 3.15 Rennovation and Remarks

In [ ]:
real_estate_df["Renovation"].unique()

array([nan, 'Done', 'Not yet'], dtype=object)

In [ ]:
real_estate_df.groupby("Type")["Renovation"].value_counts(dropna=False)

Type                                 Renovation
Agricultural Land                    NaN            9300
Forest Land                          NaN            5004
Pre-owned Condominiums, etc.         Not yet       20668
                                     Done           9320
                                     NaN            2467
Residential Land(Land Only)          NaN           59233
Residential Land(Land and Building)  NaN           78556
Name: count, dtype: int64

Renovation data is only recorded for pre-owned condos.

In [ ]:
real_estate_df["Remarks"].unique()

array([nan, 'Dealings in which auction or arbiter participates',
       'Dealings including private road', 'Dealings of adjacent land',
       'Dealings between related objects',
       'Dealings including special circumstances',
       'Dealings between related objects,Dealings including private road',
       'Dealings in which auction or arbiter participates,Dealings including private road',
       'Dealings of real estate that includes wound etc.',
       'Dealings including valueless house',
       'Dealings of adjacent land,Dealings including private road',
       'Dealings including private road,Dealings including special circumstances',
       'Dealings of adjacent land,Dealings in which auction or arbiter participates,Dealings including private road',
       'Dealings between related objects,Dealings in which auction or arbiter participates',
       'Dealings of adjacent land,Dealings in which auction or arbiter participates',
       'Dealings of real estate to which mortgage e

In [ ]:
real_estate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184548 entries, 0 to 184547
Data columns (total 51 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   No                           184548 non-null  int64  
 1   Type                         184548 non-null  object 
 2   Region                       137789 non-null  object 
 3   MunicipalityCode             184548 non-null  int64  
 4   Prefecture                   184548 non-null  object 
 5   Municipality                 184548 non-null  object 
 6   DistrictName                 184489 non-null  object 
 7   NearestStation               184548 non-null  object 
 8   TimeToNearestStation         169166 non-null  object 
 9   MinTimeToNearestStation      169166 non-null  float64
 10  MaxTimeToNearestStation      167277 non-null  float64
 11  TradePrice                   184548 non-null  int64  
 12  FloorPlan                    31436 non-null   object 
 13 

### 4. Export Cleaned Dataset

### Engineered and Cleaned Feature Dictionary

| Feature | Description | Processing / Notes |
|---|---|---|
| `AvgTimeToNearestStation` | Average estimated time to the nearest station, in minutes | Derived from the average between the minimum and maximum station travel times; missing where unavailable |
| `TimeRangeToNearestStation` | Difference between maximum and minimum estimated station travel times, in minutes | Derived from the station travel-time bounds |
| `HasStation` | Indicator associated with the availability of station travel-time information | `1` if the relevant station travel-time field is recorded; `0` otherwise |
| `AreaSizeCategory` | Categorical representation of land area size | Derived from `Area`|
| `LandShapeReplaced` | Land-shape field after merging similar categories |
| `FrontageReplaced` | Frontage field after replacing missing values | Numeric field |
| `HasBuilding` | Indicator of whether the property type implies a building | Based on the mapping from `Type`; not determined solely by whether `TotalFloorArea` is present |
| `BuildingAge` | Estimated building age at the time of the transaction | Calculated using `Year` and `BuildingYear`|
| `StructureCleaned` | Simplified building-structure category | Derived from `Structure`; if only the first category is retained, secondary structure labels are lost |
| `StructureCleanedReplaced` | Structure field after replacing missing values |
| `UseSimple` | Simplified version of the recorded property use | Derived from `Use`|
| `PurposeImputed` | Purpose field after filling or assigning categories for missing values | this name does not necessarily mean statistical imputation |
| `CityPlanningCleaned` | City-planning field after cleaning missing or special categories |


In [ ]:
#Export data to CSV

real_estate_df.to_csv("real_estate_cleaned.csv", index=False)